In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

## Silicon Labs AI/ML Tooling X Databricks

This notebook presents the use-case of bringing a data scientist's ML modeling journey on Databricks while utilizing Silicon Labs' AI/ML tooling to profile it before spending time and resources on expensive compute to train the model.

This notebook also unifies the data scientist's journey with an embedded engineer's journey, reducing the chances of misalignment between host and embedded environments.

This notebook includes the following steps:
1. Load and preprocess data
2. Trains a teacher model, distills it into a student model, and exports the final model artifacts.  
The output includes the trained **.h5 model**, **.tflite model** for profiling.
3. Register the model in MLflow
4. Convert the registered Tensorflow Keras model to a quantized `.tflite` model
5. Profile the model *before* training using Silicon Labs' Model Profiler
6. Log profiling results using MLflow
7. Choose the model with the "right" profiling results

In [0]:
WORKSPACE_DIR = dbutils.secrets.get(scope="silabs-mlops", key="workspace-dir")

MODEL_PATH = f"{WORKSPACE_DIR}/training/vosk-model-small-en-us-0.15"
AUDIO_PATH = "/Volumes/mlops_dev/default/audio_raw"
WORKSPACE_MODELS_DIR = f"{WORKSPACE_DIR}/training/models"
Teacher_H5 = f"{WORKSPACE_MODELS_DIR}/Custom_model_teacher.h5"
DataRoot = AUDIO_PATH

#### Install Dependencies

In [0]:
%pip install silabs-mltk audiomentations noisereduce pyloudnorm librosa tensorflow scikit-learn tensorflow-model-optimization vosk

#### Install `silabs-mlops` (`sml ops`)

In [0]:
%pip install silabs-mlops

dbutils.library.restartPython()

In [0]:
import shutil
import sys
from pathlib import Path

from sml.ops.model.profiler import NPUProfiler

profiler = NPUProfiler()
try:
    installed_path = profiler.install_profiler()
    print(f"[OK] mvp_profiler installed at: {installed_path}")
except FileExistsError as e:
    print(f"[SKIP] {e}")
    exit(0)
except Exception as e:
    print(f"[FAIL] mvp_profiler installation failed: {e}", file=sys.stderr)
    exit(1)

if shutil.which("mvp_profiler") is None and shutil.which("mvp_profiler.exe") is None:
    install_dir = str(Path(installed_path).parent)
    print(f"Note: add '{install_dir}' to your PATH to run 'mvp_profiler' directly.")
    print("The SDK will also auto-detect it in ~/.sml/bin.")

In [0]:
from sml.ops import data

data.config(
    server_endpoint=dbutils.secrets.get(
        scope="silabs-mlops", key="zerobus-server-endpoint"
    ),
    workspace_url=dbutils.secrets.get(
        scope="silabs-mlops", key="zerobus-workspace-url"
    ),
    table_name=dbutils.secrets.get(scope="silabs-mlops", key="zerobus-table-name"),
    client_id=dbutils.secrets.get(scope="silabs-mlops", key="zerobus-client-id"),
    client_secret=dbutils.secrets.get(
        scope="silabs-mlops", key="zerobus-client-secret"
    ),
)

#### Preprocessing the data
Cleaning the raw audio data for removing false positives

In [0]:
import os
import sys

from vosk_preprocessing import run_refinement

print(f"--- Starting Offline Refinement on {AUDIO_PATH} ---")

# Run refinement and rename files immediately
results = run_refinement(AUDIO_PATH, MODEL_PATH, auto_rename=True)

if results:
    print(f"\nSUCCESS: Refined {len(results)} files. Updating metadata table...")

    for res in results:
        # 2. Update the metadata table in Unity Catalog
        # We update file_name, file_path, and class_label based on the refinement
        query = f"""
            UPDATE mlops_dev.default.stream_audio_metadata 
            SET file_name = '{res["new_name"]}', 
                file_path = '{res["new_path"]}', 
                class_label = '{res["refined_label"]}'
            WHERE file_name = '{res["original_name"]}'
        """
        try:
            # Execute Spark SQL update
            spark.sql(query)
            print(f" [SQL OK] Updated metadata for {res['original_name']}")
        except Exception as sql_err:
            print(
                f" [SQL ERR] Failed to update table for {res['original_name']}: {sql_err}"
            )

    print("\n--- Refinement and Metadata Sync Complete ---")
else:
    print("\nNo 'unknown' files were successfully refined at this time.")

#### Setup MLFlow

In [0]:
import mlflow

with mlflow.start_run() as run:
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")

## Train Models

#### Teacher Model
Trains the teacher model by **loading a base model if it exists**, otherwise **starting from scratch**. The `mltk_core` pipeline automatically resumes from an existing `custom_model_teacher.h5` or initializes a new model when none is found.

#### Train Student Model

Trains the student model by **loading a previous student checkpoint if available**, otherwise **starting from scratch**, and uses the fine‑tuned teacher model for knowledge distillation.

#### Register the model in MLflow

In [0]:
# Train Teacher Model
os.environ["TRAIN_TEACHER"] = "1"

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=True)

base_model_path = f"{WORKSPACE_DIR}/training/models/Custom_model_teacher.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False, weights=base_model_path)
    _, teacher_accuracy = results.get_best_metric()
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)
    _, teacher_accuracy = results.get_best_metric()

print(results)

with mlflow.start_run(run_id=run_id) as run:
    mlflow.log_metric("teacher_accuracy", teacher_accuracy)

In [0]:
# Train Student Model
import os

os.environ["TRAIN_TEACHER"] = "0"

import mltk.core as mltk_core
from custom_model import my_model, prepare_teacher_or_student_model

prepare_teacher_or_student_model(train_teacher=False)

base_model_path = f"{WORKSPACE_DIR}/training/models/Custom_model.h5"
if os.path.exists(base_model_path):
    print(f"Resuming training from: {base_model_path}")
    results = mltk_core.train_model(my_model, clean=False)
    _, student_accuracy = results.get_best_metric()
else:
    print("Starting training from scratch...")
    results = mltk_core.train_model(my_model, clean=True)
    _, student_accuracy = results.get_best_metric()

print(results)

with mlflow.start_run(run_id=run_id) as run:
    mlflow.log_metric("student_accuracy", student_accuracy)

#### Locate and Copy Model Artifacts##
This step searches for trained model artifacts inside driver nodes, identifies the correct `custom_model` folder, and copies the generated `.h5` and `.tflite` files into your Workspace for easy access.

In [0]:
import glob
import os
import pprint
import shutil

# 1) Find all potential driver homes
driver_homes = [d.rstrip("/") for d in glob.glob("/home/spark-*/")]
if not driver_homes:
    raise RuntimeError("No /home/spark-*/ found on this cluster")

# 2) For each driver, look for a non-empty custom_model folder
candidates = []
for dh in driver_homes:
    cm_dir = f"{dh}/.mltk/models/custom_model"
    if os.path.isdir(cm_dir):
        try:
            files = os.listdir(cm_dir)
        except Exception:
            files = []
        candidates.append((cm_dir, files))

print("MLTK candidate dirs:")
pprint.pp(candidates)

# 3) Choose the one that actually has artifacts (h5 or tflite)
src_dir = None
for cm_dir, files in candidates:
    if any(f.endswith((".h5", ".tflite")) for f in files):
        src_dir = cm_dir
        break

if not src_dir:
    raise RuntimeError(
        "No custom_model artifacts found. Re-run STUDENT training, then run this cell again."
    )

# 4) Copy selected artifacts to Workspace Files
# Creating a folder in workspace files to store the models Eg: models

dst_dir = f"{WORKSPACE_DIR}/training/models"
os.makedirs(dst_dir, exist_ok=True)

with mlflow.start_run(run_id=run_id) as run:
    for name in [
        "custom_model.h5",
        "custom_model.tflite",
        "custom_model.tflite.summary.txt",
    ]:
        src = os.path.join(src_dir, name)
        if os.path.exists(src):
            print("Copying:", src, "->", os.path.join(dst_dir, name))
            shutil.copy2(src, os.path.join(dst_dir, name))
            mlflow.log_artifact(src)
        else:
            print("[WARN] Missing:", src)

## Model Profiling
Profiling the models using the model.profile function by providing the compiled .tflite or .zip model path and automatically uploading profiling artifacts and history logs to Databricks Unity Catalog Volumes path specified.

Log profiling metrics, parameters, and artifacts to MLflow for tracking and comparing model profiling runs across different models and configurations. The profiling result object from the cell above is used to populate the MLflow run.

In [0]:
from sml.ops import model
import mlflow
import os

model_path = f"{WORKSPACE_DIR}/training/models/custom_model.tflite"
print("\n--- [A] Local Simulation & Cloud Upload ---")
try:
    result = model.profile(
        model_path=model_path,
        use_simulator=True,  # Set to True to run the model profiling locally using the simulator
        volume_path="/Volumes/mlops_dev/default/profiling_results",
    )
    # The result object contains all the extracted data:
    print(f"  \u2713 Model:         {result.model_name}")
    print(f"  \u2713 Arena Size:    {result.arena_size_kb:.1f} KB")
    print(f"  \u2713 Total MACs:    {result.total_macs:,}")
    print(f"  \u2713 Remote Folder: {result.output_dir}")
    print(f"  \u2713 History Log:   {result.history_log_path}")
except Exception as e:
    # If there is a failure, the script will crash here
    # but the history.log will STILL upload to the volume path.
    print(f"  [!] Profiling failed -> {e}")

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_id=run_id) as run:
    for name, value in result.raw_report.items():
        if name == "layers":
            for layer in value:
                mlflow.log_param(f"layer_{layer['name']}", layer)
        else:
            mlflow.log_param(name, value)

    if os.path.isdir(result.output_dir):
        mlflow.log_artifacts(result.output_dir)

    # Tags for easy filtering
    mlflow.set_tag("task", "model_profiling")
    mlflow.set_tag("model_type", "tflite")
    mlflow.set_tag("accelerator", result.raw_report["accelerator"])

    print(f"✓ MLflow Run ID:     {run.info.run_id}")
    print(
        f"✓ Metrics logged:    arena_size_kb={result.arena_size_kb}, total_macs={result.total_macs:,}"
    )
    print("✓ Artifacts logged:  model .tflite + profiling outputs")
    print(
        f"\n  View run: {mlflow.get_tracking_uri()}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
    )

## Model Registration

In [0]:
def register_single_file(file_path, uc_model_name, run_id):
    import mlflow
    import os
    from mlflow import MlflowClient
    import numpy as np
    from mlflow.models import infer_signature
    from mlflow.pyfunc import PythonModel

    print(f"[UC] Registering: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # Ensure no active run
    if mlflow.active_run():
        mlflow.end_run()

    # ---- Minimal pyfunc so UC accepts model directory ----
    class Identity(PythonModel):
        def predict(self, context, model_input):
            return model_input

    # Dummy signature (UC requires model signature)
    dummy_input = np.zeros((1, 1), dtype="float32")
    dummy_output = np.zeros((1, 1), dtype="float32")
    signature = infer_signature(dummy_input, dummy_output)

    # ---- START RUN ----
    with mlflow.start_run(run_id=run_id) as run:
        # Log as a pyfunc model with the file as an extra artifact
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=Identity(),
            signature=signature,
            input_example=dummy_input,
            artifacts={"model_file": file_path},
        )

        # Register the model
        mv = mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=uc_model_name,
        )

        client = MlflowClient()
        client.set_registered_model_alias(uc_model_name, "champion", mv.version)

    print(f"[UC] Registered {uc_model_name} \u2192 version {mv.version}")
    return mv.version

In [0]:
teacher_v = register_single_file(
    "/Workspace/Users/raansari@silabs.com/training/models/Custom_model_teacher.h5",
    "mlops_dev.default.Custom_model_teacher",
    run_id,
)

In [0]:
student_v = register_single_file(
    "/Workspace/Users/raansari@silabs.com/training/models/custom_model.tflite",
    "mlops_dev.default.Custom_model_student",
    run_id,
)